# Representation Geometry

How do internal representations change as information flows through the model?
This notebook explores the geometric structure of activations across layers.

Covered: CKA similarity, subspace overlap, intrinsic dimensionality, layer similarity matrices, and representation drift.

## Why This Matters

The geometry of internal representations — measured via CKA, intrinsic dimensionality, and subspace overlap — reveals how models organize information across layers. This is key for understanding whether different models learn similar representations and how representations transform through the network.

**Key references:**
- [Kornblith et al. (2019) "Similarity of Neural Network Representations"](https://arxiv.org/abs/1905.00414) — CKA for comparing representations across models and layers
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import geometry

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

In [ ]:
# Prepare diverse prompts
prompts = [
    "The capital of France is Paris, which is known for",
    "Machine learning models can be trained using gradient",
    "The quick brown fox jumped over the lazy dog and",
    "In mathematics, a prime number is a natural number",
    "The stock market experienced significant volatility during",
    "Scientists recently discovered a new species of deep sea",
    "The history of computing begins with mechanical calculators",
    "Cooking a perfect steak requires careful attention to",
    "The theory of relativity was proposed by Albert Einstein",
    "Programming languages like Python are widely used for",
]
token_seqs = [model.to_tokens(p) for p in prompts]
print(f"Prepared {len(token_seqs)} prompts")

## 1. Layer Similarity Matrix (CKA)

**Centered Kernel Alignment (CKA)** measures how similar the representations are between
any two layers. This reveals which layers compute similar features and where major
representational changes happen.

In [ ]:
sim_result = geometry.layer_similarity_matrix(model, token_seqs, method="cka")

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_result["matrix"], cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(sim_result["labels"])))
ax.set_yticks(range(len(sim_result["labels"])))
ax.set_xticklabels(sim_result["labels"], rotation=45, ha="right")
ax.set_yticklabels(sim_result["labels"])
ax.set_title("Layer Similarity (CKA)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 2. Pairwise Representational Similarity

Compare specific pairs of layers.

In [ ]:
# CKA between adjacent layers
print("CKA between adjacent layers:")
prev_hook = "hook_embed"
for l in range(model.cfg.n_layers):
    curr_hook = f"blocks.{l}.hook_resid_post"
    cka = geometry.representational_similarity(model, token_seqs, prev_hook, curr_hook)
    print(f"  {prev_hook.replace('blocks.', 'L').replace('.hook_resid_post', ''):>10s} -> L{l}: {cka:.4f}")
    prev_hook = curr_hook

In [ ]:
# CKA vs cosine similarity for early/late comparison
early = "blocks.0.hook_resid_post"
late = f"blocks.{model.cfg.n_layers - 1}.hook_resid_post"

cka_sim = geometry.representational_similarity(model, token_seqs, early, late, method="cka")
cos_sim = geometry.representational_similarity(model, token_seqs, early, late, method="cosine")
print(f"Early vs Late:  CKA={cka_sim:.4f}  Cosine={cos_sim:.4f}")

## 3. Subspace Overlap

Do the principal components of activations at different layers span the same subspace?

In [ ]:
# Subspace overlap between pairs of layers
print("Subspace overlap (top 10 PCs):")
for l in [0, 3, 6, 9, 11]:
    hook = f"blocks.{l}.hook_resid_post"
    overlap = geometry.subspace_overlap(
        model, token_seqs, "hook_embed", hook, n_dims=10
    )
    print(f"  embed vs L{l}: {overlap:.4f}")

## 4. Intrinsic Dimensionality

How many effective dimensions do the representations use at each layer?

In [ ]:
# Collect activations at each layer
layer_dims_pr = []
layer_dims_ev = []
layer_labels = []

for l in range(model.cfg.n_layers):
    hook = f"blocks.{l}.hook_resid_post"
    acts = []
    for tokens in token_seqs:
        _, cache = model.run_with_cache(tokens)
        acts.append(np.array(cache[hook])[-1])  # last position
    acts = np.stack(acts)
    
    dim_pr = geometry.intrinsic_dimensionality(acts, method="participation_ratio")
    dim_ev = geometry.intrinsic_dimensionality(acts, method="explained_variance_90")
    layer_dims_pr.append(dim_pr)
    layer_dims_ev.append(dim_ev)
    layer_labels.append(f"L{l}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(len(layer_labels)), layer_dims_pr)
axes[0].set_xticks(range(len(layer_labels)))
axes[0].set_xticklabels(layer_labels, rotation=45)
axes[0].set_ylabel("Participation Ratio")
axes[0].set_title("Intrinsic Dimensionality (Participation Ratio)")
axes[0].axhline(model.cfg.d_model, color='red', linestyle='--', alpha=0.5, label=f'd_model={model.cfg.d_model}')
axes[0].legend()

axes[1].bar(range(len(layer_labels)), layer_dims_ev, color='orange')
axes[1].set_xticks(range(len(layer_labels)))
axes[1].set_xticklabels(layer_labels, rotation=45)
axes[1].set_ylabel("Components for 90% Variance")
axes[1].set_title("Intrinsic Dimensionality (90% Explained Variance)")

plt.tight_layout()
plt.show()

## 5. Representation Drift

How much do representations change between consecutive layers?

In [ ]:
drift = geometry.representation_drift(model, token_seqs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(drift["labels"]))
axes[0].bar(x, drift["l2_distances"])
axes[0].set_xticks(x)
axes[0].set_xticklabels(drift["labels"], rotation=45, ha="right", fontsize=7)
axes[0].set_ylabel("L2 Distance")
axes[0].set_title("Representation Drift (L2)")

axes[1].bar(x, drift["cosine_similarities"], color="green")
axes[1].set_xticks(x)
axes[1].set_xticklabels(drift["labels"], rotation=45, ha="right", fontsize=7)
axes[1].set_ylabel("Cosine Similarity")
axes[1].set_title("Consecutive Layer Similarity")
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `representational_similarity()` | CKA or cosine similarity between layer representations |
| `subspace_overlap()` | Overlap between principal component subspaces |
| `intrinsic_dimensionality()` | Effective dimensionality via participation ratio or explained variance |
| `layer_similarity_matrix()` | Pairwise similarity between all layers |
| `representation_drift()` | L2 distance and cosine similarity between consecutive layers |